In [1]:
# Import requests library for making HTTP requests to websites
import requests

# Import BeautifulSoup for parsing and navigating HTML content
from bs4 import BeautifulSoup
from bs4 import builder
import quopri 

import numpy as np
# Import pandas for data manipulation and CSV export
import pandas as pd

# Import time for adding delays between requests
import time

# Import random for randomizing delays and user agents
import random

# Import datetime for timestamping our scraped data
from datetime import datetime

# Import os for file and directory management
import os

print("Libraries imported successfully")

Libraries imported successfully


In [25]:
# Path where your HTML pages are stored
folder_path = r"C:\Users\suvar\OneDrive\ESILV\WebScrapping\TourismReviews\Cairo"

# Pattern for reviewIndex files
file_pattern = os.path.join(folder_path, "index*.mhtml")

# List of all extracted reviews
all_reviews = []


BASE_URL = "https://www.tripadvisor.com/"

In [3]:

# =========================
# FUNCTION TO LOAD REVIEWS
# =========================

def Load_Reviews(start_page, end_page,  city, place,):

    all_reviews = []

    for page in range(start_page, end_page + 1):

        file_name = f"index{page}.mhtml"
        file_path = os.path.join(folder_path, file_name)

        if not os.path.exists(file_path):
            print(f"{file_name} not found. Skipping page {page}.")
            continue

        try:
            with open(file_path, "rb") as f:
                raw = f.read()

            decoded = quopri.decodestring(raw).decode("utf-8", errors="ignore")
            soup = BeautifulSoup(decoded, "html.parser")

            reviews = soup.select('[data-automation="reviewCard"]')
            print(f"Page {page}: Reviews found: {len(reviews)}")

            for review in reviews:

                title_tag = review.find("span", class_="yCeTE")
                title = title_tag.get_text(strip=True) if title_tag else "not-specified"

                date_tag = review.find("div", string=lambda x: x and "Written" in x)
                review_date = date_tag.get_text(strip=True).replace("Written ", "") if date_tag else "not-specified"

                location_tag = review.select_one("div.biGQs._P.navcl span")
                reviewer_location = location_tag.get_text(strip=True) if location_tag else "not-specified"

                rating_tag = review.select_one('svg[data-automation="bubbleRatingImage"] title')
                rating_text = rating_tag.get_text(strip=True) if rating_tag else "not-specified"
                rating = rating_text.split(" ")[0] if rating_text != "not-specified" else "not-specified"

                review_data = {
                    "City": city,
                    "Place": place,
                    "Page": page,
                    "Title": title,
                    "Review_Date": review_date,
                    "Reviewer_Location": reviewer_location,
                    "Rating": rating,
                }

                all_reviews.append(review_data)

        except Exception as e:
            print(f"Error processing page {page}: {str(e)}")

    print(f"\nTotal Reviews Extracted: {len(all_reviews)}")
    return all_reviews




In [35]:
# =========================
# LOAD NEW REVIEWS
# =========================
start_page = 153
end_page = 181
city = "Cairo"
place ="Mosque of Muhammad Ali"





csv_file = "cairo_reviews.csv"

all_reviews = Load_Reviews(start_page, end_page,city, place)

print(f"\nTotal Reviews Loaded: {len(all_reviews)}")


# =========================
# PREVIEW REVIEWS
# =========================

for i, review in enumerate(all_reviews[:5], 1):
    print(f"\nReview {i}")
    print("-" * 50)
    print("City:", review["City"])
    print("Place:", review["Place"])
    print("Page:", review["Page"])
    print("Title:", review["Title"])
    print("Date:", review["Review_Date"])
    print("Location:", review["Reviewer_Location"])
    print("Rating:", review["Rating"])




Page 153: Reviews found: 10
Page 154: Reviews found: 10
Page 155: Reviews found: 10
Page 156: Reviews found: 10
Page 157: Reviews found: 10
Page 158: Reviews found: 10
Page 159: Reviews found: 10
Page 160: Reviews found: 10
Page 161: Reviews found: 10
Page 162: Reviews found: 10
Page 163: Reviews found: 10
Page 164: Reviews found: 10
Page 165: Reviews found: 10
Page 166: Reviews found: 10
Page 167: Reviews found: 10
Page 168: Reviews found: 10
Page 169: Reviews found: 10
Page 170: Reviews found: 10
Page 171: Reviews found: 10
Page 172: Reviews found: 10
Page 173: Reviews found: 10
Page 174: Reviews found: 10
Page 175: Reviews found: 10
Page 176: Reviews found: 10
Page 177: Reviews found: 10
Page 178: Reviews found: 10
Page 179: Reviews found: 10
Page 180: Reviews found: 10
Page 181: Reviews found: 10

Total Reviews Extracted: 290

Total Reviews Loaded: 290

Review 1
--------------------------------------------------
City: Cairo
Place: Mosque of Muhammad Ali
Page: 153
Title: Impressive 

In [36]:
# =========================
# SAVE ONLY NEW PAGES TO CSV
# =========================

new_df = pd.DataFrame(all_reviews)

if os.path.exists(csv_file):

    existing_df = pd.read_csv(csv_file)

    existing_pages = set(existing_df["Page"].unique())

    new_df = new_df[~new_df["Page"].isin(existing_pages)]

    if len(new_df) == 0:
        print("\nNo new pages to save.")
    else:
        final_df = pd.concat([existing_df, new_df], ignore_index=True)

        final_df.to_csv(
            csv_file,
            index=False,
            encoding="utf-8-sig"
        )

        print(f"\nAdded {len(new_df)} new reviews")
        print(f"Total reviews in file: {len(final_df)}")

else:
    new_df.to_csv(
        csv_file,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"\nCreated file with {len(new_df)} reviews")


Added 290 new reviews
Total reviews in file: 1820
